# validation-no-grad — worked example 1: MSE validation under no_grad

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `validation-no-grad`.

**This is a worked example — read it, run each cell, and follow the reasoning.** It's study material, so there's nothing to submit here. Delta Drills hands you a hands-on version to complete yourself as you get comfortable with the idea.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

_First time on this topic? Run the **Setup** cell above and skim it: every class and helper mentioned below is defined there. You don't need to have done any other drill first._

`with t.no_grad():` disables autograd inside the block: tensors produced there have `requires_grad=False` and no `grad_fn`, so no computation graph is built. This is the standard wrapper for a validation pass, saving memory and forbidding accidental `.backward()` on eval outputs.

## Worked solution

We run a validation pass and confirm autograd is off.

1. We open `with t.no_grad():` so everything inside is graph-free.
2. We forward the model, compute the MSE `((logits - y) ** 2).mean()`, and extract the scalar with `.item()` — all inside the block.
3. Because no graph was recorded, the returned `loss` has `grad_fn is None`; calling `.backward()` on it would raise.

We print the loss value and verify `loss.requires_grad` is False, proving the block did its job.

In [ ]:
import torch as t
import torch.nn as nn

t.manual_seed(0)
model = nn.Linear(4, 4)
x = t.randn(3, 4)
y = t.randn(3, 4)

def validate(model, x, y):
    with t.no_grad():
        logits = model(x)
        loss = ((logits - y) ** 2).mean()
        return loss.item(), loss

val, loss = validate(model, x, y)
print('loss:', round(val, 4))
print('requires_grad:', loss.requires_grad)
print('grad_fn is None:', loss.grad_fn is None)